# Version 2 Process Run and Review

Run the daily preparation pipeline and inspect the main outputs. This notebook is written to work in Google Colab when `version_2` is stored in Google Drive at `/content/drive/MyDrive/version_2`, and it also falls back to `/content/version_2` or the local project path.

## Mount Drive and Set Root

Purpose: mount Google Drive in Colab, then point the notebook to the process project root.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    pass

In [ ]:
from pathlib import Path
import subprocess, sys
import pandas as pd
from IPython.display import display

root_candidates = [
    Path('/content/drive/MyDrive/version_2/process'),
    Path('/content/version_2/process'),
    Path.cwd().resolve().parents[0],
]
ROOT = next(path for path in root_candidates if path.exists())
CONFIG = ROOT / 'configs' / 'default.yaml'
ROOT

## Install Requirements

Purpose: install the small Python dependency set in Colab.


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(ROOT / 'requirements.txt')], check=True)
print('Installed:', ROOT / 'requirements.txt')


## Run Process Pipeline

Purpose: execute the full daily processing pipeline and save the outputs.


In [ ]:
cmd = [sys.executable, str(ROOT / 'run_process.py'), '--config', str(CONFIG), '--stages', 'all']
print('Running:', ' '.join(map(str, cmd)))
proc = subprocess.Popen(cmd, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
rc = proc.wait()
if rc != 0:
    raise RuntimeError(f'Process pipeline failed with code {rc}')


## Review Core Outputs

Purpose: inspect the key process summaries and the top of the daily model dataset.


In [ ]:
display(pd.read_csv(ROOT / 'outputs' / '00_validation' / 'raw_stock_summary.csv'))
display(pd.read_csv(ROOT / 'outputs' / '01_stock' / 'clean_stock_summary.csv'))
display(pd.read_csv(ROOT / 'outputs' / '02_macro' / 'macro_missing_share.csv').head(20))
display(pd.read_csv(ROOT / 'outputs' / '03_model_data' / 'daily_model_data.csv', nrows=20))
